# 03 core 피처 13개 점검

- `src/features.py`로 150행 × 13열 피처 표를 만든다
- 검증: 결측 없음, 같은 신호 두 번 → 같은 값
- 피처별 정상/고장 차이: 중앙값 비, 고장 초과율(정상 95번째 백분위 초과), 고장 미달율(정상 5번째 백분위 미만)

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.data import load_dataset
from src.features import extract_features, feature_table

X, y, names = load_dataset(ROOT / "data" / "TeamProject_train_변경")
F = feature_table(X)
print("피처 표:", F.shape)

## 1. 검증

In [ ]:
print("결측·무한대 개수:", (~np.isfinite(F.to_numpy())).sum())
a, b = extract_features(X[0]), extract_features(X[0])
print("같은 신호 두 번 → 같은 값:", all(a[k] == b[k] for k in a))
print("대역 비율 합 (1이어야 함), 최소·최대:",
      F.filter(like="_band_").sum(axis=1).min().round(6), F.filter(like="_band_").sum(axis=1).max().round(6))

## 2. 피처별 정상/고장 차이

고장에서 커지는 피처는 '초과율', 작아지는 피처는 '미달율'이 크다

In [ ]:
rows = []
for col in F.columns:
    v0, v1 = F[col][y == 0], F[col][y == 1]
    rows.append({
        "피처": col,
        "정상 중앙값": v0.median(),
        "고장 중앙값": v1.median(),
        "중앙값 비": v1.median() / v0.median(),
        "고장 초과율": (v1 > v0.quantile(0.95)).mean(),
        "고장 미달율": (v1 < v0.quantile(0.05)).mean(),
    })
print(pd.DataFrame(rows).round(3).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(15, 8))
for ax, col in zip(axes.ravel(), F.columns):
    v = F[col]
    title = col
    if v.min() > 0 and v.max() / v.min() > 100:
        v, title = np.log10(v), f"log10 {col}"  # 값 범위가 넓으면 로그를 취해 그림
    ax.boxplot([v[y == 0], v[y == 1]], tick_labels=["정상", "고장"])
    ax.set_title(title, fontsize=9)
for ax in axes.ravel()[len(F.columns):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. 피처 간 상관 (정상 120개만)

|r| ≥ 0.85 쌍은 정보가 겹친다 (`pipeline.md` 상관 정리 기준). 정상만 쓰는 이유: 고장 때문에 함께 커지는 '클래스 차이'가 상관으로 섞이지 않게

In [ ]:
corr = F[y == 0].corr(method="spearman")
pairs = [(a, b, corr.loc[a, b]) for i, a in enumerate(F.columns) for b in F.columns[i + 1:]
         if abs(corr.loc[a, b]) >= 0.85]
print("|r| ≥ 0.85 쌍 (Spearman, 정상만):")
for a, b, r in sorted(pairs, key=lambda p: -abs(p[2])):
    print(f"  {a} ~ {b}: {r:.2f}")